**Basic Setup**

In [8]:
!mv ~/Downloads/"kaggle (2).json" ~/Downloads/kaggle.json

mv: /Users/khushiteli/Downloads/kaggle (2).json: No such file or directory


In [9]:
!mkdir -p ~/.kaggle

In [10]:
!cp ~/Downloads/kaggle.json ~/.kaggle/

In [11]:
!chmod 600 ~/.kaggle/kaggle.json

In [12]:
!kaggle datasets list

ref                                                            title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
wardabilal/spotify-global-music-dataset-20092025               Spotify Global Music Dataset (2009–2025)               1289021  2025-11-11 09:43:05.933000           6195        128  1.0              
sadiajavedd/students-academic-performance-dataset              Students_Academic_Performance_Dataset                     8907  2025-10-23 04:16:35.563000          11664        291  1.0              
kundanbedmutha/instagram-analytics-dataset                     Instagram Analytics Dataset                            1090208  2025-11-19 09:28:48.650000           1385         33  1.0              
sonal

In [14]:
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri

Dataset URL: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
License(s): MIT
  0%|                                               | 0.00/86.8M [00:00<?, ?B/s]
100%|██████████████████████████████████████| 86.8M/86.8M [00:00<00:00, 4.00GB/s]


In [16]:
!unzip brain-tumor-classification-mri.zip

Archive:  brain-tumor-classification-mri.zip
  inflating: Testing/glioma_tumor/image(1).jpg  
  inflating: Testing/glioma_tumor/image(10).jpg  
  inflating: Testing/glioma_tumor/image(100).jpg  
  inflating: Testing/glioma_tumor/image(11).jpg  
  inflating: Testing/glioma_tumor/image(12).jpg  
  inflating: Testing/glioma_tumor/image(13).jpg  
  inflating: Testing/glioma_tumor/image(14).jpg  
  inflating: Testing/glioma_tumor/image(15).jpg  
  inflating: Testing/glioma_tumor/image(16).jpg  
  inflating: Testing/glioma_tumor/image(17).jpg  
  inflating: Testing/glioma_tumor/image(18).jpg  
  inflating: Testing/glioma_tumor/image(19).jpg  
  inflating: Testing/glioma_tumor/image(2).jpg  
  inflating: Testing/glioma_tumor/image(20).jpg  
  inflating: Testing/glioma_tumor/image(21).jpg  
  inflating: Testing/glioma_tumor/image(22).jpg  
  inflating: Testing/glioma_tumor/image(23).jpg  
  inflating: Testing/glioma_tumor/image(24).jpg  
  inflating: Testing/glioma_tumor/image(25).jpg  
  infl

In [17]:
!pip install timm grad-cam gradio torch torchvision matplotlib scikit-learn -q

In [18]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import timm
import gradio as gr

from sklearn.metrics import classification_report, confusion_matrix

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [25]:
DATASET_ROOT = r"/Users/khushiteli/Project_2"   

In [26]:
TRAIN_DIR = os.path.join(DATASET_ROOT, "Training")
TEST_DIR  = os.path.join(DATASET_ROOT, "Testing")

IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS     = 5
VAL_SPLIT  = 0.2   
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

**Transforms and Dataset**

In [27]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

val_test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

In [28]:
full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
test_dataset       = datasets.ImageFolder(TEST_DIR,  transform=val_test_tfms)

class_names = full_train_dataset.classes
num_classes = len(class_names)
print("Classes:", class_names)

Classes: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']


In [29]:
val_size   = int(len(full_train_dataset) * VAL_SPLIT)
train_size = len(full_train_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

print(f"Train size: {train_size}, Val size: {val_size}, Test size: {len(test_dataset)}")

Train size: 2296, Val size: 574, Test size: 394


In [30]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

**ViT and ResNet Model**

In [31]:
def build_vit(num_classes: int):
    model = timm.create_model(
        "vit_base_patch16_224",
        pretrained=True,
        num_classes=num_classes
    )
    return model

def build_resnet(num_classes: int):
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = resnet.fc.in_features
    resnet.fc = nn.Linear(in_features, num_classes)
    return resnet

**Training and Evaluation function**

In [32]:
from torch.optim import AdamW

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


In [33]:
def eval_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds)

In [34]:
def run_experiment(model, model_name, train_loader, val_loader, epochs=EPOCHS, lr=1e-4):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr)

    best_val_acc = 0
    best_state   = None

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = eval_epoch(model, val_loader, criterion)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = model.state_dict()
            torch.save(best_state, f"{model_name}_best.pth")
            print(f"  -> New best {model_name} saved (val_acc={val_acc:.4f})")

    print(f"Best {model_name} val_acc: {best_val_acc:.4f}")

**Training ViT & ResNet on the 4 types**

In [35]:
vit_model = build_vit(num_classes)
run_experiment(vit_model, "vit", train_loader, val_loader, epochs=EPOCHS)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[vit] Epoch 1/5 | train_loss=0.7685, train_acc=0.6703 | val_loss=0.5198, val_acc=0.8258
  -> New best vit saved (val_acc=0.8258)
[vit] Epoch 2/5 | train_loss=0.2918, train_acc=0.8976 | val_loss=0.2247, val_acc=0.9181
  -> New best vit saved (val_acc=0.9181)
[vit] Epoch 3/5 | train_loss=0.1962, train_acc=0.9273 | val_loss=0.2159, val_acc=0.9181
[vit] Epoch 4/5 | train_loss=0.1483, train_acc=0.9495 | val_loss=0.2357, val_acc=0.9251
  -> New best vit saved (val_acc=0.9251)
[vit] Epoch 5/5 | train_loss=0.1130, train_acc=0.9621 | val_loss=0.5692, val_acc=0.8659
Best vit val_acc: 0.9251


In [36]:
resnet_model = build_resnet(num_classes)
run_experiment(resnet_model, "resnet", train_loader, val_loader, epochs=EPOCHS)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/khushiteli/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████████████████████████████████| 97.8M/97.8M [00:01<00:00, 62.8MB/s]


[resnet] Epoch 1/5 | train_loss=0.5523, train_acc=0.7966 | val_loss=0.3362, val_acc=0.8763
  -> New best resnet saved (val_acc=0.8763)
[resnet] Epoch 2/5 | train_loss=0.1738, train_acc=0.9408 | val_loss=0.1463, val_acc=0.9512
  -> New best resnet saved (val_acc=0.9512)
[resnet] Epoch 3/5 | train_loss=0.1119, train_acc=0.9643 | val_loss=0.1143, val_acc=0.9599
  -> New best resnet saved (val_acc=0.9599)
[resnet] Epoch 4/5 | train_loss=0.1028, train_acc=0.9643 | val_loss=0.1360, val_acc=0.9495
[resnet] Epoch 5/5 | train_loss=0.0378, train_acc=0.9874 | val_loss=0.2052, val_acc=0.9321
Best resnet val_acc: 0.9599


**Evaluation**

In [37]:
def evaluate_on_test(model, model_name, test_loader):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, y_true, y_pred = eval_epoch(model, test_loader, criterion)

    print(f"{model_name} Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

vit_best = build_vit(num_classes)
vit_best.load_state_dict(torch.load("vit_best.pth", map_location=device))

resnet_best = build_resnet(num_classes)
resnet_best.load_state_dict(torch.load("resnet_best.pth", map_location=device))

evaluate_on_test(vit_best,  "ViT",    test_loader)
evaluate_on_test(resnet_best, "ResNet", test_loader)

ViT Test Loss: 1.1130, Test Acc: 0.7538

Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       0.74      0.34      0.47       100
meningioma_tumor       0.68      0.97      0.80       115
        no_tumor       0.76      0.90      0.82       105
 pituitary_tumor       0.97      0.77      0.86        74

        accuracy                           0.75       394
       macro avg       0.79      0.74      0.74       394
    weighted avg       0.77      0.75      0.73       394

Confusion Matrix:
[[ 34  43  23   0]
 [  0 112   1   2]
 [  4   7  94   0]
 [  8   3   6  57]]
ResNet Test Loss: 1.0225, Test Acc: 0.7538

Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       1.00      0.24      0.39       100
meningioma_tumor       0.67      0.98      0.80       115
        no_tumor       0.72      0.98      0.83       105
 pituitary_tumor       1.00      0.77      0.87        74

        accurac

In [38]:
single_preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

def load_image_as_tensor(img: Image.Image):
    img = img.convert("RGB")
    tensor = single_preprocess(img).unsqueeze(0)  # (1,3,H,W)
    return tensor.to(device)


In [39]:
def compute_saliency_map(model, img_tensor, target_class=None):
    """
    img_tensor: (1,3,H,W)
    returns: saliency (H,W) in [0,1], predicted_class_index
    """
    model.eval()
    img = img_tensor.clone().detach().requires_grad_(True)

    output = model(img)  # (1,num_classes)
    if target_class is None:
        target_class = output.argmax(dim=1).item()

    score = output[0, target_class]
    model.zero_grad()
    score.backward()

    saliency = img.grad.abs().max(dim=1)[0]  # (1,H,W) -> (H,W)
    saliency -= saliency.min()
    saliency /= (saliency.max() + 1e-8)
    saliency = saliency.squeeze().detach().cpu().numpy()
    return saliency, target_class


In [40]:
class GradCAM:
    def __init__(self, model, target_layer_name="layer4"):
        self.model = model
        self.model.eval()
        self.gradients = None
        self.activations = None

        modules = dict([*self.model.named_modules()])
        self.target_layer = modules[target_layer_name]

        self.target_layer.register_forward_hook(self._forward_hook)
        # newer PyTorch:
        self.target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, img_tensor, target_class=None):
        """
        img_tensor: (1,3,H,W)
        returns: cam (H,W) in [0,1], predicted_class_index
        """
        img = img_tensor.clone().detach().requires_grad_(True)
        output = self.model(img)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        score = output[0, target_class]
        self.model.zero_grad()
        score.backward(retain_graph=True)

        gradients   = self.gradients    # (B,C,H,W)
        activations = self.activations  # (B,C,H,W)

        alpha = gradients.mean(dim=(2, 3), keepdim=True)           # (B,C,1,1)
        weighted = (alpha * activations).sum(dim=1, keepdim=True)  # (B,1,H,W)
        cam = F.relu(weighted)
        cam = F.interpolate(cam, size=img.shape[2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0]  # (H,W)

        cam -= cam.min()
        cam /= (cam.max() + 1e-8)
        cam = cam.detach().cpu().numpy()
        return cam, target_class


In [41]:
def overlay_heatmap_on_image(img: Image.Image, heatmap: np.ndarray, alpha=0.4):
    img = img.resize((heatmap.shape[1], heatmap.shape[0]))
    img_np = np.array(img).astype(np.float32) / 255.0

    cmap = plt.get_cmap("jet")
    heatmap_color = cmap(heatmap)[:, :, :3]  # (H,W,3)

    overlay = (1 - alpha) * img_np + alpha * heatmap_color
    overlay = np.clip(overlay, 0, 1)
    overlay = (overlay * 255).astype(np.uint8)
    return Image.fromarray(overlay)


In [42]:
vit_model = build_vit(num_classes)
vit_model.load_state_dict(torch.load("vit_best.pth", map_location=device))
vit_model.to(device)
vit_model.eval()

resnet_model = build_resnet(num_classes)
resnet_model.load_state_dict(torch.load("resnet_best.pth", map_location=device))
resnet_model.to(device)
resnet_model.eval()

resnet_cam = GradCAM(resnet_model, target_layer_name="layer4")


In [43]:
def predict_and_explain(image: Image.Image, model_choice: str):
    if image is None:
        return "Please upload an image.", None

    img_tensor = load_image_as_tensor(image)

    if model_choice == "ViT":
        model = vit_model
        with torch.no_grad():
            outputs = model(img_tensor)
            probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
            pred_idx = int(probs.argmax())
            pred_class = class_names[pred_idx]
            confidence = float(probs[pred_idx])

        saliency, _ = compute_saliency_map(model, img_tensor, target_class=pred_idx)
        overlay = overlay_heatmap_on_image(image, saliency)

    else:  # ResNet
        model = resnet_model
        with torch.no_grad():
            outputs = model(img_tensor)
            probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()
            pred_idx = int(probs.argmax())
            pred_class = class_names[pred_idx]
            confidence = float(probs[pred_idx])

        cam, _ = resnet_cam.generate(img_tensor, target_class=pred_idx)
        overlay = overlay_heatmap_on_image(image, cam)

    text = f"Prediction: {pred_class} (confidence: {confidence:.2f})"
    return text, overlay

In [46]:
with gr.Blocks() as demo:
    gr.Markdown("""
    # AI Radiologist Assistant (Tumor Type Classification)

    Upload a brain MRI image from your dataset and choose a model (ViT or ResNet).  
    The assistant will:
    - Predict **which tumor type** it is (4 classes)
    - Show a heatmap highlighting regions that influenced the decision
    """)

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Upload MRI Image")
            model_choice = gr.Radio(
                choices=["ViT", "ResNet"],
                value="ViT",
                label="Choose model"
            )
            submit_btn = gr.Button("Analyze")
        with gr.Column():
            output_text  = gr.Textbox(label="Prediction")
            output_image = gr.Image(label="Explanation Heatmap")

    submit_btn.click(
        fn=predict_and_explain,
        inputs=[input_image, model_choice],
        outputs=[output_text, output_image]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
